# Q-Learning Agent on FrozenLake-v1
**Assignment:** Custom Reinforcement Learning Agent using Farama Gymnasium  
**Environment:** `FrozenLake-v1` (4×4, deterministic with `is_slippery=False`)  
**Algorithm:** Tabular Q-learning with epsilon-greedy exploration.

Run all cells from top to bottom. The notebook writes the Q-table, reward plot, and GIF into `outputs/`.

In [ ]:
!pip -q install "gymnasium[toy-text]>=1.0.0" numpy matplotlib pillow

## 1. Imports and environment initialization
Gymnasium provides the standard environment API: `reset()` begins an episode and `step()` applies an action.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from IPython.display import display, Image as IPyImage

SEED = 42
os.makedirs("outputs", exist_ok=True)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
env.action_space.seed(SEED)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Actions: 0=Left, 1=Down, 2=Right, 3=Up")
print("Map: S=start, F=frozen, H=hole, G=goal")

## 2. Q-learning agent
The Q-table has one row per state and one column per action. The update moves the current estimate toward the observed reward plus the discounted best next-state value.

In [ ]:
class QLearningAgent:
    def __init__(self, n_states, n_actions, lr=0.8, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.02, epsilon_decay=0.0005, seed=42):
        self.q = np.zeros((n_states, n_actions), dtype=float)
        self.lr, self.gamma = lr, gamma
        self.epsilon, self.epsilon_min = epsilon, epsilon_min
        self.epsilon_decay = epsilon_decay
        self.rng = np.random.default_rng(seed)

    def act(self, state, explore=True):
        if explore and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.q.shape[1]))
        return int(np.argmax(self.q[state]))

    def learn(self, s, a, r, ns, done):
        target = r if done else r + self.gamma * np.max(self.q[ns])
        self.q[s, a] += self.lr * (target - self.q[s, a])

    def decay(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * (1 - self.epsilon_decay))

agent = QLearningAgent(env.observation_space.n, env.action_space.n, seed=SEED)
print("Q-table shape:", agent.q.shape)

## 3. Train the agent
Each episode resets the environment. The agent explores early and gradually shifts toward greedy actions as epsilon decreases.

In [ ]:
EPISODES = 20_000
episode_rewards, episode_lengths = [], []
for ep in range(EPISODES):
    state, _ = env.reset(seed=SEED + ep)
    total_reward, steps, done = 0.0, 0, False
    while not done and steps < 100:
        action = agent.act(state, explore=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.learn(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1
    agent.decay()
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

np.save("outputs/q_table.npy", agent.q)
np.save("outputs/episode_rewards.npy", np.asarray(episode_rewards))
print("Training episodes:", len(episode_rewards))
print("Final epsilon:", round(agent.epsilon, 4))
print("Q-table saved to outputs/q_table.npy")

## 4. Plot convergence
A moving average smooths the episode-to-episode variation. In this task, reward is 1 when the agent reaches the goal and 0 otherwise.

In [ ]:
window = 100
ma = np.convolve(episode_rewards, np.ones(window)/window, mode="valid")
plt.figure(figsize=(9, 4.5))
plt.plot(episode_rewards, alpha=0.2, label="Episode reward")
plt.plot(np.arange(len(ma)) + window - 1, ma, label="100-episode moving average", linewidth=2)
plt.xlabel("Episode"); plt.ylabel("Reward"); plt.title("Q-learning training progress")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig("outputs/training_curve.png", dpi=160)
plt.show()

## 5. Evaluate the learned greedy policy
Evaluation disables exploration. Success rate is the fraction of episodes that reach the goal.

In [ ]:
successes, lengths = 0, []
for ep in range(500):
    state, _ = env.reset(seed=50_000 + ep)
    done, steps, final_reward = False, 0, 0
    while not done and steps < 100:
        action = int(np.argmax(agent.q[state]))
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        final_reward = reward
        steps += 1
    successes += int(final_reward > 0)
    lengths.append(steps)
success_rate = successes / 500
print(f"Greedy policy success rate: {success_rate:.3f} ({successes}/500)")
print(f"Mean episode length: {np.mean(lengths):.2f} steps")
print("Greedy actions for states 0–15:", np.argmax(agent.q, axis=1))

## 6. Hyperparameter sensitivity
Compare learning rates using the same environment and seed. Each setting is trained independently and evaluated greedily. Because tabular learning is stochastic, results may vary slightly between runs.

In [ ]:
def run_setting(lr, gamma=0.95, episodes=5000, seed=7):
    a = QLearningAgent(env.observation_space.n, env.action_space.n, lr=lr,
                       gamma=gamma, epsilon=1.0, epsilon_min=0.02,
                       epsilon_decay=0.001, seed=seed)
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        done, steps = False, 0
        while not done and steps < 100:
            act = a.act(s, True)
            ns, r, term, trunc, _ = env.step(act)
            done = term or trunc
            a.learn(s, act, r, ns, done)
            s = ns; steps += 1
        a.decay()
    wins = 0
    for ep in range(200):
        s, _ = env.reset(seed=70_000 + ep)
        done, steps, r = False, 0, 0
        while not done and steps < 100:
            s, r, term, trunc, _ = env.step(int(np.argmax(a.q[s])))
            done = term or trunc; steps += 1
        wins += int(r > 0)
    return wins / 200

learning_rates = [0.1, 0.4, 0.8, 1.0]
rates = [run_setting(lr) for lr in learning_rates]
for lr, rate in zip(learning_rates, rates):
    print(f"learning_rate={lr:.1f} -> success_rate={rate:.3f}")
plt.figure(figsize=(6, 4))
plt.bar([str(x) for x in learning_rates], rates)
plt.ylim(0, 1.05); plt.xlabel("Learning rate"); plt.ylabel("Greedy success rate")
plt.title("Hyperparameter sensitivity: learning rate")
plt.tight_layout(); plt.savefig("outputs/hyperparameter_sensitivity.png", dpi=160); plt.show()

## 7. Save a short GIF of the agent
This cell records the environment's actual rendered frames while the greedy policy acts. The GIF will be saved as `outputs/agent_demo.gif`.

In [ ]:
from PIL import Image
render_env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")
state, _ = render_env.reset(seed=123)
frames = [Image.fromarray(render_env.render())]
done, steps = False, 0
while not done and steps < 100:
    action = int(np.argmax(agent.q[state]))
    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    frames.append(Image.fromarray(render_env.render()))
    steps += 1
frames[0].save("outputs/agent_demo.gif", save_all=True, append_images=frames[1:], duration=450, loop=0)
print("GIF saved to outputs/agent_demo.gif; reward:", reward, "steps:", steps)
display(IPyImage(filename="outputs/agent_demo.gif"))
render_env.close()
env.close()

## 8. Final checklist
- [x] Environment initialized with `gymnasium.make()`
- [x] Q-learning implemented from scratch (no RL algorithm library)
- [x] Training reward curve generated
- [x] Greedy policy evaluated
- [x] Learning-rate sensitivity tested
- [x] Q-table saved as `.npy`
- [x] GIF saved as `.gif`

For the final submission, include the complete repository and the PDF report. Use the actual success rates printed by your own execution in the report.